In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

df = pd.read_csv("/content/algerian3.csv")

df.columns = df.columns.str.strip()
df['Classes'] = df['Classes'].str.strip()

df['Target'] = df['Classes'].map({'fire': 1, 'not fire': 0})

X = df.drop(columns= ['Classes', 'day', 'month', 'year'])
y = df['Target']

X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = y.loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model =keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='elu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='elu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation="sigmoid"),
])
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_21 (Dense)                │ (None, 16)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 337 (1.32 KB)

 Trainable params: 337 (1.32 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss',
                               patience=3,
                               restore_best_weights=True)

In [ ]:
history = model.fit(X_train, y_train,
                    validation_split=0.2,
                    epochs=100,
                    batch_size=4,
                    verbose=2,
                    callbacks=[early_stopping])

Epoch 1/100
39/39 - 2s - 52ms/step - accuracy: 0.5419 - loss: 0.8204 - val_accuracy: 0.2308 - val_loss: 0.9067
Epoch 2/100
39/39 - 0s - 7ms/step - accuracy: 0.5097 - loss: 0.7880 - val_accuracy: 0.3333 - val_loss: 0.8315
Epoch 3/100
39/39 - 0s - 10ms/step - accuracy: 0.5548 - loss: 0.7304 - val_accuracy: 0.4103 - val_loss: 0.7653
Epoch 4/100
39/39 - 0s - 4ms/step - accuracy: 0.6323 - loss: 0.6526 - val_accuracy: 0.4872 - val_loss: 0.7023
Epoch 5/100
39/39 - 0s - 4ms/step - accuracy: 0.7161 - loss: 0.6058 - val_accuracy: 0.5641 - val_loss: 0.6486
Epoch 6/100
39/39 - 0s - 5ms/step - accuracy: 0.6774 - loss: 0.5713 - val_accuracy: 0.6410 - val_loss: 0.5979
Epoch 7/100
39/39 - 0s - 4ms/step - accuracy: 0.7161 - loss: 0.5554 - val_accuracy: 0.6667 - val_loss: 0.5542
Epoch 8/100
39/39 - 0s - 4ms/step - accuracy: 0.7548 - loss: 0.4933 - val_accuracy: 0.7949 - val_loss: 0.5153
Epoch 9/100
39/39 - 0s - 4ms/step - accuracy: 0.7935 - loss: 0.4709 - val_accuracy: 0.8462 - val_loss: 0.4817
Epoch 10

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 0.0197

Test Accuracy: 100.00%
